# Natural-Flow Adequacy Prediction in Solution-Gas-Drive Reservoirs

## Supervisor-facing notebook guide
This notebook is the first executable modeling stage. It demonstrates loading, canonicalizing, auditing, engineering lagged history features, defining a transparent natural-flow threshold event, and evaluating fixed-horizon oil-rate prediction on held-out wells. The included data are synthetic and only verify that the workflow runs. No performance number from the synthetic sample is evidence of field accuracy.

### Research objective
Estimate future production and pressure behavior from reservoir type, generalized place/region, history, and current operating/reservoir measurements. The current prototype focuses on rate forecasting and event construction. A validated time-to-inadequacy probability model requires field records with documented lift status and operational/economic threshold.

### Running cell by cell
Run each cell in order. Set `DATA_PATH` to your file and complete `COLUMN_MAP` and `UNIT_MAP` before using field data. Accepted files are CSV, XLSX/XLS, Parquet, and JSON. Arbitrary file formats cannot be interpreted safely without a parser and a column/unit mapping.

## 1. Environment and project imports
The import cell loads the local project package. If dependencies are missing, install `requirements.txt` from the repository root before continuing.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from sgd_ml.io import load_table
from sgd_ml.features import normalize_columns, engineer_history
from sgd_ml.labels import add_forward_targets, label_inadequacy
from sgd_ml.modeling import split_by_well, split_by_time, make_regressor, evaluate_regression

## 2. Data source and explicit mappings
A canonical table requires `well_id`, `date`, and `oil_rate`. Column mapping renames source columns. Record units separately; mapping a column name does not convert units. Geography should be generalized if exact location is sensitive.

In [ ]:
DATA_PATH = PROJECT_ROOT / "data/synthetic/demo_well_monthly.csv"
COLUMN_MAP = {
    "year": "year",
    "month": "month",
}
UNIT_MAP = {"oil_rate": "volume/day", "gas_rate": "volume/day", "water_rate": "volume/day", "reservoir_pressure": "pressure", "bhp": "pressure"}
raw = load_table(DATA_PATH)
if {"year", "month"}.issubset(raw.columns) and "date" not in raw.columns:
    raw["date"] = pd.to_datetime(raw["year"].astype(str) + "-" + raw["month"].astype(str) + "-01", errors="coerce")
print(f"Loaded {len(raw):,} observations from {raw['well_id'].nunique():,} wells")
raw.head()

## 3. Schema and quality audit
The audit reports missingness and duplicate well-date records before feature construction. Duplicate observations need engineering resolution, not blind averaging. Check whether the production frequency is regular and whether rates are measured, allocated, or estimated.

In [ ]:
required = {"well_id", "date", "oil_rate"}
missing = required.difference(raw.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")
duplicate_count = raw.duplicated(["well_id", "date"]).sum()
quality = pd.DataFrame({"missing_count": raw.isna().sum(), "missing_fraction": raw.isna().mean()})
print(f"Duplicate well-date rows: {duplicate_count}")
display(quality.sort_values("missing_fraction", ascending=False))

## 4. Canonicalizing columns and checking measurement coverage
Replace the illustrative mapping with the source dataset's verified names. The following line expects canonical names to already exist in the sample. Use explicit mappings for a field file and convert units before combining sources.

In [ ]:
df = normalize_columns(raw)
for col in ["oil_rate", "gas_rate", "water_rate", "reservoir_pressure", "bhp", "water_cut", "gor"]:
    if col in df:
        df[col] = pd.to_numeric(df[col], errors="coerce")
print("Well count:", df.well_id.nunique())
print("Time span:", df.date.min().date(), "to", df.date.max().date())
display(df.groupby("reservoir_type", dropna=False).agg(wells=("well_id", "nunique"), rows=("well_id", "size")) if "reservoir_type" in df else df.groupby("well_id").size().describe())

## 5. Engineering lagged production and pressure features
Features use values available up to the observation time. Shifted rolling summaries avoid including the current target measurement in historical predictors. Calendar and elapsed-time variables support trend context. For irregular measurements, resample to a documented cadence before applying period windows.

In [ ]:
featured = engineer_history(df, windows=(3, 6, 12))
print("Engineered table:", featured.shape)
featured.filter(regex="(lag1|mean3|slope3|pressure_minus_bubble_point)$").head()

## 6. Defining the inadequacy event
The 10-unit/day value below is an example only. The team must select and document a threshold using operating constraints and economic assumptions. Two consecutive periods reduce single-period noise but do not solve data-quality or shut-in issues. Shut-ins and lift-active periods are excluded from the natural-flow interpretation.

In [ ]:
NATURAL_FLOW_THRESHOLD = 10.0
CONSECUTIVE_PERIODS = 2
labeled = label_inadequacy(featured, threshold=NATURAL_FLOW_THRESHOLD, min_consecutive=CONSECUTIVE_PERIODS)
print("Rows marked at first sustained threshold crossing:", int(labeled.inadequate_now.sum()))

## 7. Defining forecast labels and leakage controls
This example predicts oil rate three observations ahead. For monthly data this means three months; confirm cadence. Features at the forecast origin must not include future measurements, post-transition values, future cumulative totals, or target-derived columns.

In [ ]:
HORIZON = 3
labeled = add_forward_targets(labeled, horizons=(HORIZON,))
target = f"oil_rate_future_{HORIZON}m"
excluded = {"date", "well_id", "inadequate_now", target}
feature_cols = [c for c in labeled.select_dtypes(include=["number", "object", "bool"]).columns if c not in excluded and not c.startswith(("oil_rate_future", "reservoir_pressure_future", "months_to_inadequate"))]
model_data = labeled.dropna(subset=[target]).copy()
print("Candidate predictors including current production and pressure:", feature_cols)
print("Eligible forecast-origin rows:", len(model_data))

## 8. Holding out complete wells
The primary split evaluates transfer to wells not used in training. It does not replace a chronological holdout for future forecasting within known wells. With few wells, repeat grouped cross-validation and report uncertainty across splits. Never randomly split monthly rows.

In [ ]:
train_df, test_df = split_by_well(model_data, test_size=0.25, seed=42)
X_train, y_train = train_df[feature_cols], train_df[target]
X_test, y_test = test_df[feature_cols], test_df[target]
model = make_regressor(X_train)
model.fit(X_train, y_train)
pred = model.predict(X_test)
metrics = evaluate_regression(y_test, pred)
metrics_table = pd.DataFrame([metrics]).assign(train_wells=train_df.well_id.nunique(), test_wells=test_df.well_id.nunique(), test_rows=len(test_df))
display(metrics_table)

## 9.1. Forecasting later periods for already-known wells
The chronological split trains on earlier dates and tests on a later date block. Each well may appear on both sides, but test observations occur after the training cutoff. This approximates future prediction for existing wells and complements, rather than replaces, the unseen-well split. A production deployment would retrain only with historical records available at that origin.

In [ ]:
time_train, time_test = split_by_time(model_data, test_fraction=0.2)
time_model = make_regressor(time_train[feature_cols])
time_model.fit(time_train[feature_cols], time_train[target])
time_pred = time_model.predict(time_test[feature_cols])
time_metrics = evaluate_regression(time_test[target], time_pred)
time_metrics_table = pd.DataFrame([time_metrics]).assign(train_wells=time_train.well_id.nunique(), test_wells=time_test.well_id.nunique(), test_rows=len(time_test), cutoff=time_test.date.min().date())
display(time_metrics_table)

## 9. Forecast table and diagnostic charts
The table shows forecasts and errors in target units. Regression has no single meaningful accuracy percentage. MAE/RMSE must be compared with persistence and decline-curve baselines. R-squared can be negative on difficult holdouts and is not a standalone measure of operational usefulness.

In [ ]:
results = test_df[["well_id", "date", "reservoir_type", "region", "oil_rate", target]].copy()
results["prediction"] = pred
results["error"] = results[target] - results["prediction"]
results["absolute_error"] = results["error"].abs()
display(results.sort_values("absolute_error", ascending=False).head(20))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(y_test, pred, alpha=.55)
lo, hi = min(y_test.min(), pred.min()), max(y_test.max(), pred.max())
axes[0].plot([lo, hi], [lo, hi], "--", color="black")
axes[0].set(xlabel="Observed future oil rate", ylabel="Predicted future oil rate", title="Held-out-well forecasts")
axes[1].hist(results.error.dropna(), bins=25, edgecolor="white")
axes[1].set(xlabel="Observed minus predicted", ylabel="Count", title="Forecast error distribution")
plt.tight_layout()
plt.show()

## 10. Well-history visualization and pressure scope
These plots present observed sample histories. The starter model predicts oil rate only. Pressure forecasting must be trained and validated as a separate target using pressure measurements with units and consistent semantics; no pressure accuracy is reported in this first notebook.

In [ ]:
sample_well = df.well_id.iloc[0]
well = df[df.well_id == sample_well].sort_values("date")
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(well.date, well.oil_rate, label="Oil rate")
ax.axhline(NATURAL_FLOW_THRESHOLD, color="crimson", linestyle="--", label="Illustrative threshold")
ax.set(title=f"Observed history: {sample_well}", xlabel="Date", ylabel="Oil rate (source units)")
ax.legend(); ax.grid(alpha=.25); plt.tight_layout(); plt.show()
if "reservoir_pressure" in well:
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(well.date, well.reservoir_pressure, color="darkorange")
    ax.set(title=f"Observed pressure history: {sample_well}", xlabel="Date", ylabel="Reservoir pressure (source units)")
    ax.grid(alpha=.25); plt.tight_layout(); plt.show()

## 11. Exporting the first dashboard-style report tables
This cell writes predictions and metrics to `reports/`. These tables are reproducible outputs, not field recommendations. Add prediction intervals, subgroup metrics, drift checks, and a documented threshold review before presenting project conclusions.

In [ ]:
report_dir = PROJECT_ROOT / "reports"
report_dir.mkdir(exist_ok=True)
results.to_csv(report_dir / "demo_predictions.csv", index=False)
metrics_table.to_csv(report_dir / "demo_metrics.csv", index=False)
print("Saved demonstration outputs under", report_dir)

## 12. Interpretation and next validation requirements
The demonstration metrics only indicate whether code executed on synthetic data. They must not appear in project conclusions as model accuracy. Before model acceptance, the team should assemble licensed field histories, document reservoir drive mechanism, normalize units, define natural-flow failure/threshold and censored wells, record interventions and shut-ins, test unseen-well and forward-time splits, compare against persistence and engineering decline baselines, quantify intervals/calibration, and review performance by reservoir type and region. If 30 eligible well histories cannot be sourced, report the limitation rather than inflating the sample with synthetic wells.